# Imports and setup

In [2]:
import pandas as pd
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer,SFTConfig
from transformers import TextStreamer

/home/akanish/Deep_Learning/unsloth/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
WARNING 01-26 11:12:16 [interface.py:465] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.


In [4]:
max_seq_length = 512
lora_rank=16

In [ ]:
model,tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    max_lora_rank=16
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    lora_alpha=lora_rank*2,
    use_gradient_checkpointing="unsloth",
    random_state=3407
)

# Data Processing

In [ ]:
dataset = pd.read_csv("data/dataset.csv")

In [ ]:
dataset = Dataset.from_pandas(dataset,preserve_index=False)

In [ ]:
dataset

In [ ]:
dataset['Input_Task'][0]

In [ ]:
x = list(dataset['Reasoning_Prompts'])

In [ ]:
x = map(lambda l: len(l),x)

In [ ]:
max_gen_length = max(list(x))

In [ ]:
max_gen_length

In [ ]:
x = list(dataset['Input_Task'])

In [ ]:
x = map(lambda l: len(l),x)

In [ ]:
max_inp_length = max(list(x))

In [ ]:
max_inp_length

In [22]:
reasoning_start = "<think>"
reasoning_end = "</think>"

system_prompt = f"""
You are given with a task to solve,

Think about this task and provide the steps to solve this task.

Place your answer between {reasoning_start} and {reasoning_end}

"""

In [ ]:
system_prompt

In [ ]:
chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"

In [ ]:
chat_template = chat_template.replace("{system_prompt}",system_prompt).replace("{reasoning_start}",reasoning_start)

In [ ]:
tokenizer.chat_template = chat_template

In [ ]:
def format_dataset(x):
    reasoning = x['Reasoning_Prompts']
    task = x['Input_Task']

    final_prompt = reasoning_start + reasoning + reasoning_end

    return [
        {"role":"system","content":system_prompt},
        {"role":"user","content":task},
        {"role":"assistant","content":final_prompt}
    ]

In [ ]:
dataset['messages'] = dataset.apply(format_dataset,axis=1)

In [ ]:
dataset['text'] = tokenizer.apply_chat_template(dataset['messages'].values.tolist(),tokenize=False)

In [ ]:
dataset = Dataset.from_pandas(dataset,preserve_index=False)

# Finetuning(SFT)

In [ ]:
train_args=SFTConfig(
    dataset_text_field="text",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    warmup_steps=5,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=2,
    optim="adamw_8bit",
    weight_decay=0.001,
    lr_scheduler_type='linear',
    seed=3407,
    report_to='none'
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=train_args
)

In [ ]:
trainer.train()

In [ ]:
content = [{'content': 'You are given with a task to solve,Think about this task and provide the steps to solve this task.Place your answer between {reasoning_start} and {reasoning_end}','role':'system'},
 {'content': 'Rewrite the Safety Protocols section in contract.docx and then set font to Arial.',
  'role': 'user'}]

In [ ]:
text = tokenizer.apply_chat_template(
    content,
    tokenize=False,
    add_generation_prompt=True
)

In [ ]:
_ = model.generate(
    **tokenizer(text,return_tensors='pt').to('cuda'),
    temperature=0.8,
    max_new_tokens=2048,
    streamer=TextStreamer(tokenizer,skip_prompt=False)
)

In [ ]:
model.save_pretrained('sft_model')
tokenizer.save_pretrained('sft_tokenizer')

# Inference

In [5]:
model,tokenizer = FastLanguageModel.from_pretrained(
    model_name="sft_model",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    max_lora_rank=16
)

==((====))==  Unsloth 2026.1.3: Fast Qwen2 patching. Transformers: 4.57.3. vLLM: 0.13.0.
   \\   /|    NVIDIA GeForce RTX 3050 6GB Laptop GPU. Num GPUs = 1. Max memory: 6.0 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.1.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [41]:
content = [{'content': f'You are given with a task to solve,Think about this task and provide the steps to solve this task.Place your answer between {reasoning_start} and {reasoning_end}.','role':'system'},
 {'content': 'Rewrite the Safety Protocols section in contract.docx and then set font to Arial.',
  'role': 'user'}]

In [42]:
text = tokenizer.apply_chat_template(content,tokenize=False,add_generation_prompt=True)

In [45]:
res = model.generate(
    **tokenizer(text,return_tensors='pt').to('cuda'),
    temperature=0.8,
    max_new_tokens=2048,
    streamer=TextStreamer(tokenizer,skip_prompt=False)
)

<|im_start|>system
You are given with a task to solve,Think about this task and provide the steps to solve this task.Place your answer between <think> and </think>.<|im_end|>
<|im_start|>user
Rewrite the Safety Protocols section in contract.docx and then set font to Arial.<|im_end|>
<|im_start|>assistant
1. Use modify_text_of_doc(filename='contract.docx', topic='Safety Protocols').
2. Use modify_formatting_of_doc(filename='contract.docx', request='set font to Arial').<|im_end|>


In [54]:
tokenizer.decode(res[0]).split('<|im_start|>')[3]

"assistant\n1. Use modify_text_of_doc(filename='contract.docx', topic='Safety Protocols').\n2. Use modify_formatting_of_doc(filename='contract.docx', request='set font to Arial').<|im_end|>"